# Suno Chatterbox GPU (Colab)

1. Runtime → Change runtime type → **GPU**
2. Run all cells
3. Copy the public URL into local `suno-tts/.env` as `COLAB_TTS_URL`
4. Keep this notebook running while you play stories

In [1]:
import torch
assert torch.cuda.is_available(), "Enable GPU: Runtime → Change runtime type → GPU"
print(torch.cuda.get_device_name(0))

AssertionError: Enable GPU: Runtime → Change runtime type → GPU

In [ ]:
import os
os.chdir("/content")
!pip install -q fastapi uvicorn python-multipart
if not os.path.isdir("/content/chatterbox"):
    !git clone --depth 1 https://github.com/resemble-ai/chatterbox.git /content/chatterbox
!pip install -q -e /content/chatterbox

In [ ]:
from pathlib import Path

server = Path("/content/chatterbox_server.py")
server.write_text(r'''
from __future__ import annotations
import os, tempfile, uuid
from io import BytesIO
from pathlib import Path
from threading import Lock
import torch, torchaudio
from fastapi import FastAPI, File, Form, Header, HTTPException, UploadFile
from fastapi.responses import Response
SECRET = os.environ.get("COLAB_TTS_SECRET", "").strip()
MODEL = None
MODEL_LOCK = Lock()
app = FastAPI(title="Suno Chatterbox Colab", version="1.0.0")
def load_model():
    global MODEL
    if MODEL is not None:
        return MODEL
    from chatterbox.mtl_tts import ChatterboxMultilingualTTS
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Loading ChatterboxMultilingualTTS on {device}")
    MODEL = ChatterboxMultilingualTTS.from_pretrained(device=device)
    return MODEL
def _check_secret(x_tts_secret):
    if SECRET and (x_tts_secret or "").strip() != SECRET:
        raise HTTPException(401, "Invalid secret")
def _language_id(value):
    text = (value or "en").strip().lower()
    return "hi" if text in {"hi", "hindi", "hi-in"} else "en"
@app.get("/health")
def health():
    return {"ok": True, "cuda": torch.cuda.is_available(), "loaded": MODEL is not None}
@app.post("/generate")
async def generate(text: str = Form(...), language_id: str = Form("en"), audio: UploadFile = File(...), x_tts_secret: str | None = Header(default=None)):
    _check_secret(x_tts_secret)
    story = text.strip()
    if not story:
        raise HTTPException(400, "Text is required")
    suffix = Path(audio.filename or "prompt.wav").suffix or ".wav"
    prompt = Path(tempfile.gettempdir()) / f"suno_prompt_{uuid.uuid4().hex}{suffix}"
    prompt.write_bytes(await audio.read())
    if prompt.stat().st_size == 0:
        prompt.unlink(missing_ok=True)
        raise HTTPException(400, "Audio file is required")
    model = load_model()
    lang = _language_id(language_id)
    try:
        with MODEL_LOCK:
            wav = model.generate(story, language_id=lang, audio_prompt_path=str(prompt))
        buffer = BytesIO()
        torchaudio.save(buffer, wav.cpu(), model.sr, format="wav")
        return Response(content=buffer.getvalue(), media_type="audio/wav")
    except Exception as error:
        raise HTTPException(500, f"Chatterbox failed: {error}") from error
    finally:
        prompt.unlink(missing_ok=True)
'''.strip())
print("Wrote", server)

In [ ]:
import os, sys, threading, time
import uvicorn
os.environ["COLAB_TTS_SECRET"] = os.environ.get("COLAB_TTS_SECRET", "")
sys.path.insert(0, "/content")
from chatterbox_server import app, load_model
print("Preloading model (first run downloads weights)...")
load_model()
print("Model ready")
threading.Thread(target=lambda: uvicorn.run(app, host="0.0.0.0", port=8000, log_level="info"), daemon=True).start()
time.sleep(2)
print("API on http://127.0.0.1:8000")

In [3]:
import os, re, stat, subprocess, time, urllib.request
bin_path = "/content/cloudflared"
if not os.path.isfile(bin_path):
    urllib.request.urlretrieve(
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        bin_path,
    )
    os.chmod(bin_path, os.stat(bin_path).st_mode | stat.S_IEXEC)
proc = subprocess.Popen([bin_path, "tunnel", "--url", "http://127.0.0.1:8000", "--no-autoupdate"], stderr=subprocess.PIPE, text=True)
url = None
start = time.time()
while time.time() - start < 45:
    line = proc.stderr.readline()
    if not line:
        break
    print(line.rstrip())
    match = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
    if match:
        url = match.group(0)
        break
assert url, "Tunnel URL not found; rerun this cell"
print("\nCOLAB_TTS_URL=" + url)
print("Put that in C:\\Suno_Project\\suno-tts\\.env and restart uvicorn on port 8002")

2026-09-16T07:55:04Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-09-16T07:55:04Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-09-16T07:55:08Z INF +--------------------------------------------------------------------------------------------+
2026-09-16T07:55:08Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-09-16T07:55:08Z INF |  https://cleaner-awards-rats-louise.trycloudflare.com 